# Multi-Seed Validation — Optuna Trial #43

Run the best Optuna configuration (trial #43, Soft=2.188 µm) with 5 different seeds
to measure robustness and find the best seed.

**Runtime:** ~5 × 20 min ≈ 1.7h on T4 GPU

In [ ]:
# ============================================================
# §1  Imports + Data Loading
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import os, time, random, gc, copy
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

# --- Load data ---
DATA_DIR = Path('.')
if os.path.exists('/kaggle/working/model_input.npz'):
    DATA_DIR = Path('/kaggle/working')
elif os.path.exists('model_input.npz'):
    DATA_DIR = Path('.')
else:
    raise FileNotFoundError('model_input.npz not found.')

data = np.load(DATA_DIR / 'model_input.npz')

all_scalars         = data['mlp_input']
wf_2ch              = data['cnn_input']
targets_fund_scaled = data['targets_fund_scaled']
targets_xy          = data['targets_xy']
targets_fund        = data['targets_fund']
q_all               = data['quadrant']
u_sign_all          = data['u_sign']
r_all               = data['radius']
idx_train           = data['idx_train']
idx_test            = data['idx_test']
target_mean         = data['target_mean']
target_scale        = data['target_scale']
geom                = data['geometry']
dims                = data['dims']

X_C, Y_C, PHI0 = float(geom[0]), float(geom[1]), float(geom[2])
N_MLP_IN  = int(dims[5])
N_BINS    = int(dims[6])
N_SECTORS = int(dims[7])
N = len(q_all)
xf_all = targets_fund[:, 0]
yf_all = targets_fund[:, 1]

class _TargetScaler:
    def __init__(self, mean, scale):
        self.mean_ = mean
        self.scale_ = scale
target_scaler = _TargetScaler(target_mean, target_scale)

print(f'Data loaded: {N:,} events, MLP={N_MLP_IN}-d, CNN=2×{N_BINS}')
print(f'Train: {len(idx_train):,} | Test: {len(idx_test):,}')

In [ ]:
# ============================================================
# §2  Model + Loss (Optuna trial #43 config)
# ============================================================

U_SIGNS = torch.tensor([1.0,  1.0, -1.0, -1.0], device=device)
V_SIGNS = torch.tensor([1.0, -1.0, -1.0,  1.0], device=device)


class MultiDomainDataset(Dataset):
    def __init__(self, indices, wf_2ch, scalars_all, targets_uv,
                 targets_xy_orig, q, u_sign, augment=False):
        self.wf_2ch  = torch.tensor(wf_2ch[indices], dtype=torch.float32)
        self.scalars = torch.tensor(scalars_all[indices], dtype=torch.float32)
        self.uv      = torch.tensor(targets_uv[indices], dtype=torch.float32)
        self.xy      = torch.tensor(targets_xy_orig[indices], dtype=torch.float32)
        self.q       = torch.tensor(q[indices], dtype=torch.long)
        self.u_sign  = torch.tensor(u_sign[indices], dtype=torch.float32)
        self.augment = augment

    def __len__(self):
        return len(self.q)

    def __getitem__(self, i):
        wf = self.wf_2ch[i]
        if self.augment:
            if torch.rand(1) < 0.5:
                wf = wf + torch.randn(1, wf.shape[1]) * 0.023
            if torch.rand(1) < 0.5:
                shift = torch.randint(-3, 4, (1,)).item()
                wf = torch.roll(wf, shift, dims=-1)
            if torch.rand(1) < 0.3:
                wf = wf * (0.948 + torch.rand(1).item() * 0.104)
        return wf, self.scalars[i], self.uv[i], self.xy[i], self.q[i], self.u_sign[i]


class ResBlock1D(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(channels), nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm1d(channels))
        self.act = nn.GELU()
    def forward(self, x):
        return self.act(x + self.block(x))


class MultiDomainCNN(nn.Module):
    def __init__(self, n_mlp_in=110, n_sectors=4):
        super().__init__()
        self.cnn_stem = nn.Sequential(
            nn.Conv1d(2, 32, 7, padding=3), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2))
        self.cnn_res1 = ResBlock1D(32)
        self.cnn_mid = nn.Sequential(
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(2))
        self.cnn_res2 = ResBlock1D(64)
        self.cnn_deep = nn.Sequential(
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.GELU(), nn.MaxPool1d(2))
        self.cnn_res3 = ResBlock1D(128)
        self.cnn_head = nn.Sequential(
            nn.Conv1d(128, 128, 3, padding=1), nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(1))
        # MLP: hidden=128, dropout=0.227
        self.mlp = nn.Sequential(
            nn.Linear(n_mlp_in, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.227),
            nn.Linear(128, 128),      nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.227),
            nn.Linear(128, 128),      nn.BatchNorm1d(128), nn.GELU())
        # Fusion: dropout=0.038
        self.fusion = nn.Sequential(
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.038),
            nn.Linear(128, 128), nn.BatchNorm1d(128), nn.GELU())
        self.head_xy = nn.Sequential(
            nn.Linear(128, 96), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(96, 48), nn.GELU(), nn.Linear(48, 2))
        self.head_sector = nn.Sequential(
            nn.Linear(128, 96), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(96, 48), nn.GELU(), nn.Linear(48, n_sectors))
        self.head_usign = nn.Sequential(
            nn.Linear(128, 48), nn.GELU(), nn.Dropout(0.1), nn.Linear(48, 1))

    def forward(self, wf_2ch, scalars):
        x = self.cnn_stem(wf_2ch)
        x = self.cnn_res1(x)
        x = self.cnn_mid(x)
        x = self.cnn_res2(x)
        x = self.cnn_deep(x)
        x = self.cnn_res3(x)
        cnn_out = self.cnn_head(x).squeeze(-1)
        mlp_out = self.mlp(scalars)
        fused = self.fusion(torch.cat([cnn_out, mlp_out], dim=1))
        return self.head_xy(fused), self.head_sector(fused), self.head_usign(fused)


def log_cosh_loss(pred, target):
    diff = pred - target
    return torch.mean(torch.abs(diff) + F.softplus(-2.0 * torch.abs(diff)) - np.log(2))


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        loss = (1 - p_t) ** self.gamma * ce
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean()


def soft_unfold_torch(u_abs, v_abs, logits, xc, yc, phi0_deg, temperature=0.1):
    probs = F.softmax(logits / temperature, dim=1)
    alpha_u = (probs * U_SIGNS).sum(dim=1)
    alpha_v = (probs * V_SIGNS).sum(dim=1)
    phi_r = torch.tensor(np.radians(phi0_deg), dtype=torch.float32, device=u_abs.device)
    u_s, v_s = u_abs * alpha_u, v_abs * alpha_v
    x = xc + torch.cos(phi_r) * u_s - torch.sin(phi_r) * v_s
    y = yc + torch.sin(phi_r) * u_s + torch.cos(phi_r) * v_s
    return x, y


print('Model + Loss defined (Optuna trial #43 config).')

In [ ]:
# ============================================================
# §3  Training function (Phase 1 only, 1000 epochs)
# ============================================================

# Optuna trial #43 hyperparameters
HP = dict(
    batch_size   = 384,
    lr_max       = 1.981e-3,
    weight_decay = 1.18e-4,
    warmup_ep    = 30,
    n_cycles     = 12,
    eta_min      = 2.1e-7,
    lambda_focal = 0.855,
    lambda_soft  = 1.202,
    lambda_usign = 0.713,
    t_train      = 0.293,
    focal_gamma  = 2.17,
    focal_alpha  = [0.9, 0.9, 1.0, 1.025],
    epochs       = 1000,
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def run_phase1(seed):
    """Run full Phase 1 with given seed. Returns dict with metrics + best state_dict."""
    set_seed(seed)
    t0 = time.time()

    BATCH     = HP['batch_size']
    EPOCHS    = HP['epochs']
    LR_MAX    = HP['lr_max']
    W_DECAY   = HP['weight_decay']
    WARMUP_EP = HP['warmup_ep']
    N_CYCLES  = HP['n_cycles']
    ETA_MIN   = HP['eta_min']
    T_TRAIN   = HP['t_train']
    T_INFER   = 0.1

    LAMBDA_FUND  = 1.0
    LAMBDA_FOCAL = HP['lambda_focal']
    LAMBDA_SOFT  = HP['lambda_soft']
    LAMBDA_USIGN = HP['lambda_usign']

    T_SCALE = torch.tensor(target_scaler.scale_, dtype=torch.float32, device=device)
    T_MEAN  = torch.tensor(target_scaler.mean_,  dtype=torch.float32, device=device)

    g = torch.Generator(); g.manual_seed(seed)

    ds_train = MultiDomainDataset(idx_train, wf_2ch, all_scalars,
        targets_fund_scaled, targets_xy, q_all, u_sign_all, augment=True)
    ds_test = MultiDomainDataset(idx_test, wf_2ch, all_scalars,
        targets_fund_scaled, targets_xy, q_all, u_sign_all, augment=False)

    dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True, generator=g)
    dl_test  = DataLoader(ds_test, batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

    model = MultiDomainCNN(n_mlp_in=N_MLP_IN, n_sectors=N_SECTORS).to(device)
    focal_loss = FocalLoss(gamma=HP['focal_gamma'], alpha=HP['focal_alpha']).to(device)

    def combined_loss(pred_uv, logits, usign_logit, uv_target, xy_true, q_true, u_sign_true):
        loss_fund = log_cosh_loss(pred_uv, uv_target)
        loss_focal = focal_loss(logits, q_true)
        uv_um = pred_uv * T_SCALE + T_MEAN
        x_soft, y_soft = soft_unfold_torch(
            uv_um[:, 0], uv_um[:, 1], logits, X_C, Y_C, PHI0, temperature=T_TRAIN)
        loss_soft = 0.5 * (log_cosh_loss(x_soft, xy_true[:, 0]) +
                           log_cosh_loss(y_soft, xy_true[:, 1]))
        loss_usign = F.binary_cross_entropy_with_logits(
            usign_logit.squeeze(-1), u_sign_true)
        return (LAMBDA_FUND * loss_fund + LAMBDA_FOCAL * loss_focal +
                LAMBDA_SOFT * loss_soft + LAMBDA_USIGN * loss_usign)

    optimizer = optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=W_DECAY)
    T_0 = max(1, (EPOCHS - WARMUP_EP) // N_CYCLES)
    warmup_sched = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1e-4 / LR_MAX, end_factor=1.0,
        total_iters=WARMUP_EP * len(dl_train))
    cosine_sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T_0 * len(dl_train), T_mult=1, eta_min=ETA_MIN)
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_sched, cosine_sched],
        milestones=[WARMUP_EP * len(dl_train)])

    u_true_test = xf_all[idx_test]
    v_true_test = yf_all[idx_test]

    def evaluate():
        model.eval()
        uv_l, log_l, xy_l = [], [], []
        with torch.no_grad():
            for wf_b, sc_b, uv_b, xy_b, q_b, us_b in dl_test:
                pred_uv, logits, _ = model(wf_b.to(device), sc_b.to(device))
                uv_l.append(pred_uv.cpu()); log_l.append(logits.cpu())
                xy_l.append(xy_b)
        uv_p = torch.cat(uv_l); log_p = torch.cat(log_l); xy_t = torch.cat(xy_l)
        uv_um = uv_p.numpy() * target_scaler.scale_ + target_scaler.mean_
        fund_err = np.sqrt(((uv_um - np.column_stack([u_true_test, v_true_test]))**2).sum(1)).mean()
        acc = (log_p.argmax(1).numpy() == q_all[idx_test]).mean()
        with torch.no_grad():
            x_s, y_s = soft_unfold_torch(
                torch.tensor(uv_um[:, 0], device=device),
                torch.tensor(uv_um[:, 1], device=device),
                log_p.to(device), X_C, Y_C, PHI0, temperature=T_INFER)
        xy_np = xy_t.numpy()
        soft_err = np.sqrt((x_s.cpu().numpy() - xy_np[:, 0])**2 +
                           (y_s.cpu().numpy() - xy_np[:, 1])**2).mean()
        return fund_err, acc, soft_err

    # --- Training loop ---
    best_soft, best_fund, best_acc, best_ep = float('inf'), 0, 0, 0
    best_state = None
    snapshots = []

    restart_epochs = [WARMUP_EP + T_0 * i for i in range(1, N_CYCLES + 1)]
    cp_epochs = set(re - 1 for re in restart_epochs)
    cp_epochs.add(EPOCHS)

    for ep in range(1, EPOCHS + 1):
        model.train()
        for wf_b, sc_b, uv_b, xy_b, q_b, us_b in dl_train:
            wf_b, sc_b = wf_b.to(device), sc_b.to(device)
            uv_b, xy_b = uv_b.to(device), xy_b.to(device)
            q_b, us_b  = q_b.to(device),  us_b.to(device)
            pred_uv, logits, usign_logit = model(wf_b, sc_b)
            loss = combined_loss(pred_uv, logits, usign_logit, uv_b, xy_b, q_b, us_b)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            scheduler.step()

        is_eval = (ep % 50 == 0) or (ep in cp_epochs)
        if is_eval:
            fund_err, acc, soft_err = evaluate()
            if soft_err < best_soft:
                best_soft, best_fund, best_acc, best_ep = soft_err, fund_err, acc, ep
                best_state = copy.deepcopy(model.state_dict())
            if ep in cp_epochs:
                snapshots.append((ep, fund_err, acc, soft_err))
                print(f'  seed={seed} ep={ep:4d}  Fund={fund_err:.3f}  '
                      f'Acc={acc:.1%}  Soft={soft_err:.3f}')

    dt = time.time() - t0

    # Save best model
    save_dir = '/kaggle/working/' if os.path.exists('/kaggle') else './'
    torch.save(best_state, f'{save_dir}best_model_seed{seed}.pt')

    del model, optimizer, scheduler, dl_train, dl_test
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'seed': seed, 'soft': best_soft, 'fund': best_fund,
        'acc': best_acc, 'best_ep': best_ep, 'time': dt,
        'snapshots': snapshots
    }


print('Training function ready.')

In [ ]:
# ============================================================
# §4  Run 5 seeds
# ============================================================

SEEDS = [42, 123, 2026, 7, 314]
results = []

print(f'Multi-seed validation: {len(SEEDS)} seeds')
print(f'Config: Optuna trial #43 (Soft=2.188 µm @ seed=42)')
print('=' * 60)

for seed in SEEDS:
    print(f'\n--- Seed {seed} ---')
    res = run_phase1(seed)
    results.append(res)
    print(f'  → Soft={res["soft"]:.3f}  Fund={res["fund"]:.3f}  '
          f'Acc={res["acc"]:.1%}  ep={res["best_ep"]}  ({res["time"]:.0f}s)')

print('\n' + '=' * 60)
softs = [r['soft'] for r in results]
funds = [r['fund'] for r in results]
accs  = [r['acc'] for r in results]

print(f'Soft: {np.mean(softs):.3f} ± {np.std(softs):.3f} '
      f'[{np.min(softs):.3f}, {np.max(softs):.3f}]')
print(f'Fund: {np.mean(funds):.3f} ± {np.std(funds):.3f}')
print(f'Acc:  {np.mean(accs):.1%} ± {np.std(accs):.1%}')

best = min(results, key=lambda r: r['soft'])
print(f'\nBest seed: {best["seed"]} → Soft={best["soft"]:.3f} µm')
print(f'Model saved as: best_model_seed{best["seed"]}.pt')

# Print table
print(f'\n{"Seed":>6} {"Soft":>8} {"Fund":>8} {"Acc":>8} {"Epoch":>6} {"Time":>6}')
print('-' * 50)
for r in sorted(results, key=lambda r: r['soft']):
    marker = ' ★' if r['seed'] == best['seed'] else ''
    print(f'{r["seed"]:6d} {r["soft"]:8.3f} {r["fund"]:8.3f} '
          f'{r["acc"]:7.1%} {r["best_ep"]:6d} {r["time"]:5.0f}s{marker}')

In [ ]:
# ============================================================
# §5  Visualization
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A) Soft error per seed
ax = axes[0]
seeds_s = [str(r['seed']) for r in results]
softs_v = [r['soft'] for r in results]
colors = ['gold' if r['seed'] == best['seed'] else 'steelblue' for r in results]
ax.bar(seeds_s, softs_v, color=colors, edgecolor='navy', linewidth=0.5)
ax.axhline(np.mean(softs), color='red', ls='--', alpha=0.7,
           label=f'Mean: {np.mean(softs):.3f} ± {np.std(softs):.3f}')
ax.set_xlabel('Seed')
ax.set_ylabel('Soft error (µm)')
ax.set_title('Soft Error per Seed', fontweight='bold')
ax.legend()
for i, (s, v) in enumerate(zip(seeds_s, softs_v)):
    ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)

# B) Snapshot learning curves
ax = axes[1]
for r in results:
    eps = [s[0] for s in r['snapshots']]
    sft = [s[3] for s in r['snapshots']]
    ax.plot(eps, sft, 'o-', alpha=0.7, label=f'seed={r["seed"]} (best={r["soft"]:.3f})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Soft error (µm)')
ax.set_title('Snapshot Soft Error per Cycle', fontweight='bold')
ax.legend(fontsize=8)

plt.suptitle('Multi-Seed Validation: Optuna Trial #43',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('multiseed_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nRecommended seed for NB_Model_Training: SEED = {best["seed"]}')